<a href="https://colab.research.google.com/github/DivyaDharshiniG14/Generative-podcast/blob/main/working_podcast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install torch transformers gtts pydub gradio langdetect deep_translator


In [ ]:
from transformers import pipeline

generator = pipeline("text-generation", model="gpt2")

def generate_podcast_script(topic, genre):
    prompt = f"Generate a {genre} podcast script about {topic}."
    script = generator(prompt, max_length=500, num_return_sequences=1)
    return script[0]['generated_text']


In [ ]:
from deep_translator import GoogleTranslator

# Mapping full language names to language codes
LANGUAGE_MAP = {
    "English": "en",
    "French": "fr",
    "German": "de",
    "Spanish": "es",
    "Chinese": "zh-CN",
    # Add more languages as needed
}

def translate_text(text, target_lang):
    lang_code = LANGUAGE_MAP.get(target_lang, target_lang)  # Convert full name to code
    return GoogleTranslator(source='auto', target=lang_code).translate(text)


In [ ]:
from gtts import gTTS
import os

def text_to_speech(text, lang='en'):
    tts = gTTS(text, lang=lang)
    tts.save("podcast.mp3")
    return "podcast.mp3"


In [ ]:
from pydub import AudioSegment

def add_background_music(voice_file, bgm_file, output_file="final_podcast.mp3"):
    voice = AudioSegment.from_file(voice_file)
    bgm = AudioSegment.from_file(bgm_file).set_frame_rate(voice.frame_rate).set_channels(voice.channels)
    bgm = bgm - 20  # Reduce BGM volume by 20 dB
    mixed = voice.overlay(bgm)
    mixed.export(output_file, format="mp3")
    return output_file



In [ ]:
import gradio as gr

def generate_podcast(topic, genre, language, bgm_file):
    lang_code = LANGUAGE_MAP.get(language, language)  # Ensure it's a valid language code
    script = generate_podcast_script(topic, genre)
    translated_script = translate_text(script, lang_code)
    voice_file = text_to_speech(translated_script, lang=lang_code)  # Use language code
    final_podcast = add_background_music(voice_file, bgm_file)
    return final_podcast


gr.Interface(
    fn=generate_podcast,
    inputs=["text", "text", "text", "file"],
    outputs="audio"
).launch()
